# NanoVision Net X — Image Reconstruction (Colab)

This notebook uses:
`https://github.com/Charith003/nu_data/tree/main/nuclei_partial_annotations`

### Improvements made for sharper enhancement
- Upgraded from plain autoencoder to a **U-Net style residual denoiser** (skip connections preserve fine texture).
- Uses a mixed **L1 + SSIM loss** to reduce blur and keep structure.
- Adds optional **unsharp masking** post-processing for crisper visual output.
- Includes a **manual image upload** cell for custom testing.


In [ ]:
# Install dependencies (Colab-safe)
!pip -q install scikit-image tensorflow pillow


In [ ]:
# Clone requested dataset
!rm -rf nu_data
!git clone https://github.com/Charith003/nu_data.git

from pathlib import Path
root = Path("nu_data/nuclei_partial_annotations")
print("Dataset root:", root.resolve())
print("Exists:", root.exists())


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.filters import unsharp_mask

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:
# Load images recursively from nuclei_partial_annotations
DATA_ROOT = "nu_data/nuclei_partial_annotations"
TARGET_SIZE = (128, 128)
MAX_IMAGES = 2500

valid_ext = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
exclude_tokens = {"mask", "label", "annotation", "annot"}

all_files = []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    for fn in filenames:
        ext = os.path.splitext(fn)[1].lower()
        if ext in valid_ext:
            low = fn.lower()
            if not any(tok in low for tok in exclude_tokens):
                all_files.append(os.path.join(dirpath, fn))

all_files = sorted(all_files)[:MAX_IMAGES]
if len(all_files) == 0:
    raise RuntimeError("No usable images found in nuclei_partial_annotations.")

print(f"Using {len(all_files)} images")

def load_gray(path, target_size=(128, 128)):
    img = Image.open(path).convert("L").resize(target_size)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

images = np.stack([load_gray(p, TARGET_SIZE) for p in all_files], axis=0)
images = np.expand_dims(images, axis=-1)

perm = np.random.permutation(len(images))
images = images[perm]
split = int(0.8 * len(images))
x_train = images[:split]
x_test = images[split:]

print("x_train:", x_train.shape, "x_test:", x_test.shape)


In [ ]:
# Degradation model: moderate gaussian noise + slight blur-like corruption
def degrade(x, noise_factor=0.18):
    noisy = x + noise_factor * np.random.normal(0, 1, size=x.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = degrade(x_train)
x_test_noisy = degrade(x_test)


In [ ]:
# Sharper model: Residual U-Net denoiser
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def build_resunet(input_shape=(128, 128, 1)):
    inp = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inp, 32)
    p1 = layers.MaxPooling2D(2)(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D(2)(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D(2)(c3)

    # Bottleneck
    b = conv_block(p3, 256)

    # Decoder
    u3 = layers.UpSampling2D(2)(b)
    u3 = layers.Concatenate()([u3, c3])
    c4 = conv_block(u3, 128)

    u2 = layers.UpSampling2D(2)(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2, 64)

    u1 = layers.UpSampling2D(2)(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = conv_block(u1, 32)

    pred = layers.Conv2D(1, 1, activation="sigmoid", padding="same")(c6)

    # Residual refinement: predicted clean image added to input detail map
    out = layers.Add()([pred, inp])
    out = layers.Lambda(lambda z: tf.clip_by_value(z, 0.0, 1.0))(out)

    return models.Model(inp, out, name="NanoVisionNetX_ResUNet")

# Mixed sharpness-friendly loss
def l1_ssim_loss(y_true, y_pred):
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim_term = 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.7 * l1 + 0.3 * ssim_term

model = build_resunet(input_shape=(*TARGET_SIZE, 1))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=l1_ssim_loss)
model.summary()


In [ ]:
# Train
EPOCHS = 12
BATCH_SIZE = 8

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
]

history = model.fit(
    x_train_noisy,
    x_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Evaluate and visualize (with optional unsharp enhancement)
reconstructed = model.predict(x_test_noisy, verbose=0)

def apply_unsharp(batch, radius=1.2, amount=1.3):
    out = []
    for i in range(len(batch)):
        out.append(unsharp_mask(batch[i, ..., 0], radius=radius, amount=amount, preserve_range=True))
    out = np.array(out, dtype=np.float32)[..., None]
    return np.clip(out, 0.0, 1.0)

reconstructed_sharp = apply_unsharp(reconstructed)

subset = min(300, len(x_test))
def metric_report(pred, name):
    p_scores, s_scores = [], []
    for i in range(subset):
        gt = x_test[i, ..., 0]
        pr = pred[i, ..., 0]
        p_scores.append(psnr(gt, pr, data_range=1.0))
        s_scores.append(ssim(gt, pr, data_range=1.0))
    print(f"{name} -> PSNR: {np.mean(p_scores):.2f} dB | SSIM: {np.mean(s_scores):.4f}")

metric_report(reconstructed, "Model output")
metric_report(reconstructed_sharp, "Model output + unsharp")

num_images = min(5, len(x_test))
idx = np.random.choice(len(x_test), size=num_images, replace=False)

plt.figure(figsize=(16, 8))
for i, j in enumerate(idx):
    plt.subplot(4, num_images, i + 1)
    plt.imshow(x_test_noisy[j].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    plt.subplot(4, num_images, i + 1 + num_images)
    plt.imshow(reconstructed[j].squeeze(), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

    plt.subplot(4, num_images, i + 1 + 2 * num_images)
    plt.imshow(reconstructed_sharp[j].squeeze(), cmap="gray")
    plt.title("Enhanced")
    plt.axis("off")

    plt.subplot(4, num_images, i + 1 + 3 * num_images)
    plt.imshow(x_test[j].squeeze(), cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

plt.suptitle("NanoVision Net X Reconstruction (Sharper Enhancement)", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Manual upload test: run enhancement on your own image
try:
    from google.colab import files
    uploaded = files.upload()
except Exception as e:
    uploaded = {}
    print("Upload UI works in Google Colab only.")

if uploaded:
    fname = next(iter(uploaded.keys()))
    print("Processing:", fname)
    img = Image.open(fname).convert("L").resize(TARGET_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    inp = arr[None, ..., None]

    pred = model.predict(inp, verbose=0)
    pred_sharp = apply_unsharp(pred)

    plt.figure(figsize=(12, 3))
    plt.subplot(1, 3, 1)
    plt.imshow(arr, cmap="gray")
    plt.title("Uploaded")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(pred[0, ..., 0], cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_sharp[0, ..., 0], cmap="gray")
    plt.title("Enhanced")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    os.makedirs("reconstruction_outputs", exist_ok=True)
    Image.fromarray((pred_sharp[0, ..., 0] * 255).astype(np.uint8)).save("reconstruction_outputs/manual_upload_enhanced.png")
    print("Saved: reconstruction_outputs/manual_upload_enhanced.png")


In [ ]:
# Save model + sample artifact
os.makedirs("reconstruction_outputs", exist_ok=True)
model.save("reconstruction_outputs/nanovision_resunet_nuclei_partial_annotations.keras")
plt.imsave("reconstruction_outputs/sample_enhanced.png", reconstructed_sharp[0, ..., 0], cmap="gray")
print("Saved:")
print("- reconstruction_outputs/nanovision_resunet_nuclei_partial_annotations.keras")
print("- reconstruction_outputs/sample_enhanced.png")
